In [1]:
# -*- coding: utf-8 -*-
"""enriquecer_dataset.ipynb

Enriquecedor del dataset de queries médicas. Toma cada query del JSON generado
por data_sintetica.py y agrega un campo `enriquecimiento` con posibles
padecimientos, acción recomendada, pasos accionables y mensaje al paciente.

Diseñado para Google Colab. Cada bloque marcado con `# %%` es una celda.
Reanudable: si se corta el runtime, retoma donde se quedó.
"""

'enriquecer_dataset.ipynb\n \nEnriquecedor del dataset de queries médicas. Toma cada query del JSON generado\npor data_sintetica.py y agrega un campo `enriquecimiento` con posibles\npadecimientos, acción recomendada, pasos accionables y mensaje al paciente.\n \nDiseñado para Google Colab. Cada bloque marcado con `# %%` es una celda.\nReanudable: si se corta el runtime, retoma donde se quedó.\n'

In [2]:
# %% [1] Instalación de dependencias
!pip install anthropic --quiet

# Celda 1.5: montar Drive y definir rutas persistentes
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
BASE = Path('/content/drive/MyDrive/fundamentos_eval_rag')
BASE.mkdir(parents=True, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 12.9 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
# %% [2] Imports y configuración del cliente
import anthropic
import json
import time
import re
from pathlib import Path
from collections import Counter

# En Colab usa Secrets (panel izquierdo > 🔑) con el nombre ANTHROPIC_API_KEY
try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    import os
    API_KEY = os.environ.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic(api_key=API_KEY)

# Modelo: Opus 4.6 para mayor calidad clínica.
# Si quieres ahorrar costo, usa "claude-sonnet-4-6" o "claude-haiku-4-5"
MODELO = "claude-opus-4-6"

INPUT_PATH = BASE / "dataset_rag_medico_es.json"
OUTPUT_PATH = BASE / "dataset_rag_medico_es_enriched.json"
CHECKPOINT_PATH = BASE / "dataset_rag_medico_es_enriched_checkpoint.json"
SAVE_EVERY = 10  # guardar el JSON cada N items enriquecidos

SYSTEM_PROMPT = """Eres un médico general. Esta respuesta forma parte de una app de \
primeros auxilios offline en LATAM. La capa crítica (RCP, Heimlich, hemorragia severa) \
ya está cubierta por protocolos hardcodeados, así que tú recibes consultas NO críticas \
que requieren orientación.

Devuelves EXCLUSIVAMENTE JSON válido, sin markdown ni texto adicional, con esta estructura:

{
  "posibles_padecimientos": [
    {"nombre": "string", "prob": "alta|media|baja"}
  ],
  "accion": "autocuidado | medico_general | especialista | urgencias_inmediatas",
  "que_hacer_ahora": ["paso accionable 1", "paso 2", "paso 3"],
  "que_NO_hacer": ["string"],
  "signos_de_alarma": ["string"],
  "especialidad_sugerida": "string",
  "mensaje_paciente": "1-2 oraciones cálidas dirigidas al paciente respetando su registro emocional"
}

Reglas duras:
1. 2-4 padecimientos máximo, ordenados por probabilidad descendente. Solo `nombre` y `prob`.
2. `que_hacer_ahora`: 3-5 pasos CONCRETOS y accionables. Verbos en imperativo. No teoría.
3. Si detectas señal de emergencia real (dolor torácico opresivo + esfuerzo, déficit \
neurológico súbito, sangrado severo, dificultad respiratoria aguda, alteración de la \
conciencia, anafilaxia), accion = "urgencias_inmediatas" y `que_hacer_ahora[0]` debe \
ser "Llama a emergencias ahora mismo".
4. `que_NO_hacer`: solo si hay errores comunes que la gente comete. Si no aplica, lista vacía [].
5. `mensaje_paciente`: máximo 250 caracteres. Respeta el registro emocional sin \
minimizar ni alarmar de más.
6. Nunca prescribas dosis ni nombres comerciales de medicamentos.
7. NO incluyas el disclaimer en el JSON: la app lo muestra siempre en la UI."""


def construir_user_message(item: dict) -> str:
    return (
        f"query: {item['query']}\n"
        f"area: {item['area']}\n"
        f"demografia: {item['demografia']}\n"
        f"sistemas: {', '.join(item['sistemas'])}\n"
        f"complejidad: {item['complejidad']}\n"
        f"registro: {item['registro']}\n\n"
        "Devuelve el JSON estructurado."
    )


def extraer_json(texto: str):
    """Intenta parsear el texto como objeto JSON, limpiando casos comunes."""
    t = texto.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t)
        t = re.sub(r"\s*```$", "", t)
    inicio = t.find("{")
    fin = t.rfind("}")
    if inicio != -1 and fin != -1:
        t = t[inicio : fin + 1]
    return json.loads(t)


def enriquecer_uno(item: dict, reintentos: int = 3):
    """Llama a Claude para enriquecer un item. Devuelve el dict de enriquecimiento."""
    user_msg = construir_user_message(item)
    for intento in range(1, reintentos + 1):
        try:
            resp = client.messages.create(
                model=MODELO,
                max_tokens=2000,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}],
            )
            texto = resp.content[0].text
            data = extraer_json(texto)
            if not isinstance(data, dict):
                raise ValueError("La respuesta no es un objeto JSON")
            # Validación mínima de estructura
            campos_obligatorios = {"posibles_padecimientos", "accion", "que_hacer_ahora"}
            if not campos_obligatorios.issubset(data.keys()):
                raise ValueError(f"Faltan campos: {campos_obligatorios - data.keys()}")
            return data
        except json.JSONDecodeError as e:
            print(f"     ⚠️ JSON inválido (intento {intento}): {e}")
            time.sleep(2)
        except Exception as e:
            print(f"     ⚠️ Error API (intento {intento}): {e}")
            time.sleep(5)
    print(f"     ❌ Falló tras {reintentos} intentos: {item['query'][:60]}...")
    return None


def guardar_dataset(resultados):
    """Persiste el dataset enriquecido al archivo principal."""
    OUTPUT_PATH.write_text(
        json.dumps(resultados, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def guardar_checkpoint(indices_hechos):
    """Guarda los índices ya procesados para poder reanudar."""
    CHECKPOINT_PATH.write_text(
        json.dumps(sorted(indices_hechos), ensure_ascii=False),
        encoding="utf-8",
    )


def cargar_estado(dataset_original):
    """Lee el dataset enriquecido y checkpoint si existen."""
    if OUTPUT_PATH.exists():
        try:
            resultados = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            resultados = [dict(item) for item in dataset_original]
    else:
        resultados = [dict(item) for item in dataset_original]

    if CHECKPOINT_PATH.exists():
        try:
            indices_hechos = set(json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8")))
        except json.JSONDecodeError:
            indices_hechos = set()
    else:
        # Detectar items ya enriquecidos viendo el archivo de output
        indices_hechos = {i for i, x in enumerate(resultados) if x.get("enriquecimiento")}

    return resultados, indices_hechos

In [4]:
# %% [3] Cargar el dataset de entrada
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"No encontré {INPUT_PATH}. Corre primero data_sintetica.py "
        f"o sube el dataset a esa ruta."
    )

dataset_original = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
print(f"📂 Dataset cargado: {len(dataset_original)} queries")

📂 Dataset cargado: 500 queries


In [5]:
# %% [4] Loop principal con checkpoint y autosave
def ejecutar():
    resultados, indices_hechos = cargar_estado(dataset_original)
    desde_ultimo_guardado = 0
    total = len(dataset_original)

    if indices_hechos:
        print(f"📂 Reanudando: {len(indices_hechos)}/{total} ya enriquecidos")

    for i, item in enumerate(dataset_original):
        if i in indices_hechos:
            continue

        # Log cada 20 items para no saturar
        if i % 20 == 0 or i < 3:
            print(f"\n[{i+1}/{total}] {item['area']} | {item['tipo']}")
            print(f"  → query: {item['query'][:80]}...")

        enriquecimiento = enriquecer_uno(item)
        if enriquecimiento is not None:
            resultados[i] = {**item, "enriquecimiento": enriquecimiento}
            indices_hechos.add(i)
            desde_ultimo_guardado += 1

            if desde_ultimo_guardado >= SAVE_EVERY:
                guardar_dataset(resultados)
                guardar_checkpoint(indices_hechos)
                desde_ultimo_guardado = 0
                print(f"     💾 autosave: {len(indices_hechos)}/{total} enriquecidos")
        else:
            # Marcar como intentado para no quedar en loop infinito; pero NO en checkpoint
            # para que un futuro re-run lo reintente.
            resultados[i] = {**item, "enriquecimiento": None, "error": "max_reintentos"}

        time.sleep(0.5)  # cortesía con el rate limit

    # Guardado final
    guardar_dataset(resultados)
    guardar_checkpoint(indices_hechos)
    print(f"\n✅ Enriquecimiento completo: {len(indices_hechos)}/{total} → {OUTPUT_PATH}")
    return resultados


resultados = ejecutar()



[1/500] Medicina general / familiar | sintomas_primera_persona
  → query: desde hace 3 días me duele la cabeza por las tardes y siento como mareo cuando m...

[2/500] Medicina general / familiar | sintomas_primera_persona
  → query: mi mamá tiene 71 años y últimamente se le olvidan mucho las cosas, ayer no recor...

[3/500] Medicina general / familiar | sintomas_primera_persona
  → query: tengo una tos seca que no me deja dormir, ya van casi dos semanas, no tengo fieb...
     💾 autosave: 10/500 enriquecidos
     💾 autosave: 20/500 enriquecidos

[21/500] Medicina general / familiar | medicamentos
  → query: tomé ibuprofeno hace una hora y siento como que me arde el estómago horrible, ¿e...
     💾 autosave: 30/500 enriquecidos
     💾 autosave: 40/500 enriquecidos

[41/500] Medicina general / familiar | condiciones_diagnosticadas
  → query: ¿las personas con hipotiroidismo pueden hacer ejercicio intenso?...
     💾 autosave: 50/500 enriquecidos
     💾 autosave: 60/500 enriquecidos

[61/50

In [ ]:
# %% [5] Auditoría del output enriquecido
def auditar(resultados):
    print(f"\n📊 Total: {len(resultados)}")

    ok = [r for r in resultados if r.get("enriquecimiento")]
    fail = [r for r in resultados if not r.get("enriquecimiento")]
    print(f"   Enriquecidos OK: {len(ok)}")
    print(f"   Fallos: {len(fail)}")

    if not ok:
        return

    print("\nPor acción recomendada:")
    for k, n in Counter(r["enriquecimiento"]["accion"] for r in ok).most_common():
        print(f"  {k}: {n}")

    print("\nPor especialidad sugerida (top 10):")
    for k, n in Counter(
        r["enriquecimiento"]["especialidad_sugerida"] for r in ok
    ).most_common(10):
        print(f"  {k}: {n}")

    # Items críticos que deberían existir
    urgencias = [r for r in ok if r["enriquecimiento"]["accion"] == "urgencias_inmediatas"]
    print(f"\n🚨 Casos marcados urgencias_inmediatas: {len(urgencias)}")
    if urgencias:
        print("   (Revisar manualmente que tengan sentido — falsos positivos son peligrosos)")
        for r in urgencias[:3]:
            print(f"   - {r['area']}: {r['query'][:80]}...")

    # Items donde no se generaron pasos
    sin_pasos = [r for r in ok if not r["enriquecimiento"].get("que_hacer_ahora")]
    if sin_pasos:
        print(f"\n⚠️ Items sin pasos accionables: {len(sin_pasos)} (revisar manualmente)")

    # Distribución de cantidad de padecimientos sugeridos
    n_pad = Counter(len(r["enriquecimiento"]["posibles_padecimientos"]) for r in ok)
    print(f"\nCantidad de padecimientos sugeridos por item:")
    for k in sorted(n_pad):
        print(f"  {k}: {n_pad[k]}")

    print("\n--- 2 ejemplos al azar ---")
    import random
    for ej in random.sample(ok, min(2, len(ok))):
        print(f"\nQuery: {ej['query']}")
        print(json.dumps(ej["enriquecimiento"], ensure_ascii=False, indent=2))


auditar(resultados)